# 01 · Data audit

**Frozen snapshot:** September 16, 2026 · **Cutoff:** September 12, 2026 · **Spec:** [`docs/analysis_spec.md`](../docs/analysis_spec.md)

This notebook checks the data-quality gates in plan section 8 and records the evidence behind each cleaning rule. It never looks at an Auburn or Florida outcome. The team sections count plays only, so the situation cells can be chosen before any result is seen.

Re-run from the project root after `python -m src.ingest`, `python -m src.clean`, and `python -m src.reconcile`.

In [1]:
import json, sys
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config
from src.ingest import cfbd_dest

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 80)

RAW_GLOB = str(config.RAW_DIR / "sportsdataverse" / config.SDV_PBP_RELEASE / "play_by_play_*.parquet")
con = duckdb.connect()
con.sql(f"CREATE VIEW raw AS SELECT * FROM read_parquet('{RAW_GLOB}', union_by_name=true)")
plays = pd.read_parquet(config.PROCESSED_DIR / "plays.parquet")
gates = {}

## 1. Snapshot and schema

Every raw file is hashed in the manifest. All six seasons must load and carry every required column.

In [2]:
manifest = pd.read_csv(config.MANIFEST_PATH)
display(manifest.groupby("source").agg(files=("local_path", "size"), megabytes=("bytes", lambda b: round(b.sum() / 1e6, 1))))

import pyarrow.parquet as pq
schema = []
for season in config.ALL_SEASONS:
    meta = pq.read_metadata(config.RAW_DIR / "sportsdataverse" / config.SDV_PBP_RELEASE / config.SDV_PBP_ASSET.format(season=season))
    missing = sorted(set(config.REQUIRED_PBP_COLUMNS) - set(meta.schema.names))
    schema.append({"season": season, "rows": meta.num_rows, "columns": meta.num_columns, "missing required": ", ".join(missing) or "none"})
schema = pd.DataFrame(schema)
display(schema)
gates["All six seasons load with every required column"] = (schema["missing required"] == "none").all()

,files,megabytes
source,,
cfbd,43,25.5
official,8,6.7
sportsdataverse,7,572.1


,season,rows,columns,missing required
0,2021,163444,362,none
1,2022,252307,362,none
2,2023,254090,362,none
3,2024,277048,362,none
4,2025,293200,362,none
5,2026,58764,362,none


## 2. Keys and duplicates

`id_play` is an 18-digit identifier stored as a floating-point number, so its last digits are lost and it cannot serve as a unique key. The cleaned table uses `play_key` (game ID plus the row's position in the raw file). Exact duplicate rows are removed using the content key in `src/clean.py`.

In [3]:
keys = con.sql('''
    SELECT year AS season, count(*) AS rows, count(DISTINCT id_play) AS distinct_id_play,
           count(*) - count(DISTINCT (game_id, period, clock_minutes, clock_seconds, pos_team, down, distance,
                                      yards_to_goal, play_type, play_text)) AS duplicate_rows
    FROM raw GROUP BY 1 ORDER BY 1''').df()
display(keys)
gates["play_key unique after cleaning"] = plays.play_key.is_unique

,season,rows,distinct_id_play,duplicate_rows
0,2021,163444,86381,5399
1,2022,252307,106886,1404
2,2023,254090,102461,435
3,2024,277048,163778,499
4,2025,293200,230529,1245
5,2026,58764,58726,23


## 3. Value metric: one model across every season?

If one model scored every season, identical situations would have identical values. cfbfastR's expected points shift sharply in 2026, the only season built with its new 3.0 model. CFBD PPA is identical to three decimals in every season, turnovers included. **The analysis uses PPA** (spec section 4).

In [4]:
ep = con.sql("""
    SELECT 'cfbfastR EP, 1st & 10 at own 25, Q1, tied' AS check, year AS season, round(avg(ep_before), 2) AS v
    FROM raw WHERE down=1 AND distance=10 AND yards_to_goal=75 AND period=1 AND pos_score_diff_start=0 AND (rush=1 OR pass=1) GROUP BY year
    UNION ALL
    SELECT 'cfbfastR EPA, incompletion 1st & 10 at own 25', year, round(avg(EPA), 2)
    FROM raw WHERE play_type='Pass Incompletion' AND down=1 AND distance=10 AND yards_to_goal=75 AND period=1 AND pos_score_diff_start=0 AND NOT penalty_flag GROUP BY year
    UNION ALL
    SELECT 'cfbfastR EPA, incompletion 1st & 10 at midfield', year, round(avg(EPA), 2)
    FROM raw WHERE play_type='Pass Incompletion' AND down=1 AND distance=10 AND yards_to_goal=50 AND period=1 AND pos_score_diff_start=0 AND NOT penalty_flag GROUP BY year
    UNION ALL
    SELECT 'CFBD PPA, incompletion 1st & 10 at own 25', year, round(avg(ppa), 3)
    FROM raw WHERE play_type='Pass Incompletion' AND down=1 AND distance=10 AND yards_to_goal=75 AND period=1 AND pos_score_diff_start=0 AND NOT penalty_flag GROUP BY year
    UNION ALL
    SELECT 'CFBD PPA, incompletion 1st & 10 at midfield', year, round(avg(ppa), 3)
    FROM raw WHERE play_type='Pass Incompletion' AND down=1 AND distance=10 AND yards_to_goal=50 AND period=1 AND pos_score_diff_start=0 AND NOT penalty_flag GROUP BY year
    UNION ALL
    SELECT 'CFBD PPA, turnover on downs at opp 2 (4th & 2 incompletion)', year, round(avg(ppa), 3)
    FROM raw WHERE play_type='Pass Incompletion' AND down=4 AND distance=2 AND yards_to_goal=2 AND period<=3 AND NOT penalty_flag GROUP BY year
    UNION ALL
    SELECT 'CFBD PPA spread within the same state (should be 0)', year, round(max(sd), 3) FROM (
        SELECT year, stddev(ppa) sd FROM raw WHERE play_type='Pass Incompletion' AND NOT penalty_flag AND period=1
        AND down=1 AND distance=10 AND yards_to_goal IN (75, 50) AND pos_score_diff_start=0 GROUP BY year, yards_to_goal) GROUP BY year
""").df().pivot(index="check", columns="season", values="v")
display(ep)
ppa_rows = ep.loc[[i for i in ep.index if i.startswith("CFBD PPA,")]]
gates["PPA identical for identical states across 2021-2026"] = bool((ppa_rows.nunique(axis=1) == 1).all())

season,2021,2022,2023,2024,2025,2026
check,,,,,,
CFBD PPA spread within the same state (should be 0),0.000,0.000,0.000,0.000,0.000,0.000
"CFBD PPA, incompletion 1st & 10 at midfield",-1.114,-1.114,-1.114,-1.114,-1.114,-1.114
"CFBD PPA, incompletion 1st & 10 at own 25",-0.703,-0.703,-0.703,-0.703,-0.703,-0.703
"CFBD PPA, turnover on downs at opp 2 (4th & 2 incompletion)",-4.149,-4.149,-4.149,-4.149,-4.149,-4.149
"cfbfastR EP, 1st & 10 at own 25, Q1, tied",0.900,0.930,0.890,0.880,0.920,0.790
"cfbfastR EPA, incompletion 1st & 10 at midfield",-0.790,-0.790,-0.800,-0.790,-0.780,-0.380
"cfbfastR EPA, incompletion 1st & 10 at own 25",-0.940,-0.960,-0.930,-0.930,-0.960,-0.740


In [5]:
wp = con.sql("""
    SELECT year AS season, round(avg(wp_before), 3) AS wp_home_favored_by_7_at_kickoff
    FROM raw WHERE down=1 AND distance=10 AND yards_to_goal=75 AND period=1 AND clock_minutes>=14
      AND pos_score_diff_start=0 AND round(spread)=-7 AND pos_team=home GROUP BY 1 ORDER BY 1""").df()
display(wp)  # the WP model also changed, so garbage time uses the score-margin definition

,season,wp_home_favored_by_7_at_kickoff
0,2021,0.492
1,2022,0.492
2,2023,0.491
3,2024,0.491
4,2025,0.493
5,2026,0.576


## 4. Cleaning waterfall

Plays removed by each rule, applied in order (spec section 5). The line "of which run or pass was read from the play text" counts snaps cfbfastR typed as a fumble, safety, or post-play penalty without a run or pass flag. It jumps in the 2025 and 2026 builds, and without the recovery rule those lost fumbles would drop out of 2026.

In [6]:
waterfall = pd.read_csv(config.TABLES_DIR / "cleaning_waterfall.csv", index_col="step")
display(waterfall)

,2021,2022,2023,2024,2025,2026,total
step,,,,,,,
raw rows,163444,252307,254090,277048,293200,58764,1298853
game found in CFBD,163444,252307,254090,277048,293200,58764,1298853
"completed FBS game, regular or postseason, within cutoff",163444,160327,159011,162772,166057,32425,844036
after removing duplicate rows,158045,159415,158759,162472,165525,32414,836630
run or pass snaps (sacks count as pass),119861,120630,119537,120700,123614,23973,628315
of which run or pass was read from the play text,305,275,180,151,684,297,1892
minus no-play penalties,119712,120477,119505,120629,123529,23937,627789
minus offsetting penalties,119712,120477,119505,120629,123529,23935,627787
minus accepted penalties on the play,118255,119162,118375,119656,121677,23527,620652


In [7]:
runs, passes = plays[plays.play_family == "run"], plays[plays.play_family == "pass"]
kneels = pd.DataFrame({
    "runs": runs.groupby("season").size(), "kneels": runs.groupby("season").is_kneel.sum(),
    "passes": passes.groupby("season").size(), "spikes": passes.groupby("season").is_spike.sum(),
})
kneels["kneels per 1,000 runs"] = (1000 * kneels.kneels / kneels.runs).round(1)
kneels["spikes per 1,000 passes"] = (1000 * kneels.spikes / kneels.passes).round(1)
display(kneels)

,runs,kneels,passes,spikes,"kneels per 1,000 runs","spikes per 1,000 passes"
season,,,,,,
2021,61093,624,58768,0,10.2,0.0
2022,60591,754,60039,0,12.4,0.0
2023,60300,1029,59237,0,17.1,0.0
2024,61392,1076,59308,0,17.5,0.0
2025,62927,1102,60687,74,17.5,1.2
2026,12465,174,11508,21,14.0,1.8


Kneel detection is consistent from 2023 onward, at about 17 per 1,000 runs. The 2021–2022 text is terser, and even after recognizing "TEAM run for a loss" late in a half, those seasons detect only 10–12 per 1,000. A few hundred kneels probably remain in each of those two training seasons. They are short, low-value runs at the end of halves, and the 2023–2025 sensitivity baseline (robustness check 1) excludes those seasons entirely. Spikes are recorded in the text only from 2025 onward, so earlier seasons keep them. At the 2025 rate that is roughly 70 per season, negligible next to about 118,000 snaps.

The explosive target is not distorted by penalties inside the model sample. Checked against cfbfastR's play-only yardage, the 20-yard flag would change for 1 of 89,907 gains in 2023, 5 of 93,969 in 2025, and 1 of 18,279 in 2026.

## 5. Missing values

Raw missingness of every required field among run and pass snaps. The plan drops any field missing for more than 10% of a current team's plays.

In [8]:
fields = ["down", "distance", "yards_to_goal", "half_seconds_remaining", "score_diff", "home_spread", "ppa", "yards_gained"]
snaps = plays[~plays.is_no_play]
miss = snaps.groupby("season")[fields].apply(lambda d: d.isna().mean().mul(100).round(2))
display(miss.rename(columns={"home_spread": "spread (after CFBD backfill)"}))

current = snaps[snaps.season == config.CURRENT_SEASON]
team = pd.DataFrame({
    "Auburn offense": current[current.pos_team == config.OFFENSE_TEAM][fields].isna().mean().mul(100).round(2),
    "Florida defense": current[current.def_pos_team == config.DEFENSE_TEAM][fields].isna().mean().mul(100).round(2),
})
display(team)
gates["No required field >10% missing for Auburn offense or Florida defense"] = bool((team <= 10).all().all())

,down,distance,yards_to_goal,half_seconds_remaining,score_diff,spread (after CFBD backfill),ppa,yards_gained
season,,,,,,,,
2021,0.00,0.00,0.0,0.0,0.0,0.00,0.47,0.0
2022,0.01,0.01,0.0,0.0,0.0,0.42,0.41,0.0
2023,0.00,0.00,0.0,0.0,0.0,0.32,0.31,0.0
2024,0.00,0.00,0.0,0.0,0.0,1.65,0.55,0.0
2025,0.00,0.00,0.0,0.0,0.0,0.00,0.30,0.0
2026,0.00,0.00,0.0,0.0,0.0,0.00,0.52,0.0


,Auburn offense,Florida defense
down,0.00,0.0
distance,0.00,0.0
yards_to_goal,0.00,0.0
half_seconds_remaining,0.00,0.0
score_diff,0.00,0.0
home_spread,0.00,0.0
ppa,1.24,0.0
yards_gained,0.00,0.0


## 6. Spread coverage and range

The spread is an optional feature. Coverage matters, and so does range. Florida's game against Campbell (FCS) carries a 51.5-point spread, far into the tail of the training data.

In [9]:
model = plays[plays.in_model_sample]
cover = model.groupby("season").agg(
    plays=("play_key", "size"),
    spread_missing_pct=("spread_missing", lambda s: round(100 * s.mean(), 1)),
    from_cfbd_lines_pct=("spread_source", lambda s: round(100 * (s == "cfbd_lines").mean(), 2)),
)
display(cover)

train = model[model.season.isin(config.HISTORICAL_SEASONS) & ~model.spread_missing]
matchups = (plays[(plays.season == config.CURRENT_SEASON) & ((plays.pos_team == config.OFFENSE_TEAM) | (plays.def_pos_team == config.DEFENSE_TEAM))]
            .groupby(["game_id", "home_team", "away_team"]).home_spread.first().abs().rename("abs spread").reset_index())
matchups["share of 2021-2025 plays with |spread| at least this large"] = matchups["abs spread"].map(
    lambda s: f"{100 * (train.offense_spread.abs() >= s).mean():.1f}%")
display(matchups)

,plays,spread_missing_pct,from_cfbd_lines_pct
season,,,
2021,117123,0.0,0.12
2022,117796,0.4,0.00
2023,116944,0.3,0.33
2024,117721,1.7,0.62
2025,119951,0.0,0.52
2026,23223,0.0,0.00


,game_id,home_team,away_team,abs spread,share of 2021-2025 plays with |spread| at least this large
0,401856636,Auburn,Baylor,6.5,65.7%
1,401856637,Florida,Florida Atlantic,25.5,14.3%
2,401856671,Auburn,Southern Miss,32.5,8.2%
3,401856672,Florida,Campbell,51.5,0.4%


The Campbell game sits in the extreme tail of the spread distribution. That's why robustness check 6 (leave one game out) is required for every Florida finding.

## 7. The four 2026 games reconciled

Play-by-play, CFBD, and official team box scores are compared: Auburn's from the WMT stats feed behind auburntigers.com, Florida's from the Sidearm box scores on floridagators.com. Official NCAA stats count sacks as rushes, so the play-by-play is converted to that convention before comparing.

In [10]:
rec = pd.read_csv(config.TABLES_DIR / "reconciliation_2026.csv")
order = ["points", "plays", "rushes (incl. sacks)", "pass attempts (excl. sacks)", "total yards",
         "yards on accepted penalty snaps (info)", "interceptions thrown", "fumbles lost", "sacks allowed",
         "penalties", "penalty yards", "ppa matches CFBD, other plays (share)", "ppa matches CFBD, possession-change plays (share, info)"]
def cell(r):
    if r.metric in ("points", "penalties", "penalty yards"):
        return f"{r.cfbd:g} / {r.official:g}"
    if "share" in r.metric:
        return "n/a" if pd.isna(r.play_by_play) else f"{r.play_by_play:.0%}"
    return f"{r.play_by_play:g} / {r.official:g}" if pd.notna(r.official) else f"{r.play_by_play:g}"
rec["value"] = rec.apply(cell, axis=1)
wide = rec.pivot_table(index="metric", columns=["week", "offense"], values="value", aggfunc="first").reindex(order)
print("Cells: play-by-play / official (points and penalties: CFBD / official)")
display(wide)
display(rec.status.value_counts().rename("rows by status"))

Cells: play-by-play / official (points and penalties: CFBD / official)


week                                                             1                                                 2                                    
offense                                                     Auburn     Baylor    Florida Florida Atlantic     Auburn   Campbell    Florida Southern Miss
metric                                                                                                                                                  
points                                                     17 / 17    16 / 16    66 / 66          21 / 21    43 / 43      3 / 3    52 / 52         8 / 8
plays                                                      76 / 76    99 / 99    67 / 67          90 / 90    85 / 85    63 / 63    62 / 62       57 / 58
rushes (incl. sacks)                                       41 / 41    45 / 45    41 / 41          37 / 37    51 / 51    25 / 25    30 / 30       24 / 25
pass attempts (excl. sacks)                                35 / 35    54 / 54    26 / 26          53 / 53    34 / 34    38 / 38    32 / 32       33 / 33
total yards                                              414 / 389  455 / 436  621 / 624        398 / 396  613 / 613  198 / 210  514 / 514     253 / 221
yards on accepted penalty snaps (info)                          33         11         19                3         16         -2         12            20
interceptions thrown                                         3 / 3      1 / 1      1 / 1            2 / 2      0 / 0      0 / 0      1 / 1         2 / 2
fumbles lost                                                 0 / 0      1 / 1      0 / 0            1 / 1      1 / 1      2 / 2      0 / 1         0 / 0
sacks allowed                                                5 / 5      1 / 1      1 / 0            2 / 2      1 / 1      2 / 2      0 / 0         3 / 4
penalties                                                    6 / 6      5 / 5      3 / 3            9 / 9    10 / 10      7 / 7      8 / 8       11 / 11
penalty yards                                              47 / 47    40 / 40    32 / 32          88 / 88    73 / 73    55 / 55    95 / 95       85 / 85
ppa matches CFBD, other plays (share)                         100%       100%       100%             100%       100%       100%       100%          100%
ppa matches CFBD, possession-change plays (share, info)         0%        20%         0%              17%         0%       100%         0%           33%

status
info           68
match          53
minor           8
investigate     3
Name: rows by status, dtype: int64

In [11]:
scores = rec[rec.metric == "points"]
gates["All four final scores match official records"] = bool((scores.cfbd == scores.official).all())
counts = rec[rec.metric.isin(["plays", "interceptions thrown", "pass attempts (excl. sacks)"])]
gates["Plays, pass attempts, and interceptions within 1 of official for every offense"] = bool((counts.diff_vs_reference.abs() <= 1).all())
share = rec[rec.metric == "ppa matches CFBD, other plays (share)"]
gates["Frozen PPA equals CFBD on every non-turnover play"] = bool((share.play_by_play >= 0.99).all())

**Reading the reconciliation**

- **Points:** all four final scores match CFBD and the official team records.
- **Counts:** plays, rushes, pass attempts, and interceptions match the official box scores exactly for seven of eight offenses. Southern Miss is one rush short.
- **Fumbles:** lost fumbles match everywhere except Florida's offense against Campbell. Official "fumbles lost" also counts fumbles on kick and punt returns, which are not offensive snaps.
- **Yards:** the remaining gaps (Auburn +25 and Baylor +19 in Week 1, Southern Miss +32 in Week 2) are mostly yards on accepted-penalty snaps, where `yards_gained` includes the penalty (33, 11, and 20 yards). Those snaps are outside the model sample. What remains is 12 yards or less per offense.
- **PPA:** the frozen PPA equals CFBD's live `/plays` PPA on every non-turnover play in all four games. It differs on interceptions, lost fumbles, and turnovers on downs because CFBD has since revalued those plays. The frozen values are internally consistent (section 3), so they stay. CFBD's live PPA totals are approximate cross-checks only.
- **Gaps before the fix:** the first reconciliation was missing six real snaps across the Auburn–Baylor and Auburn–Southern Miss games, including J. Cobb's 19-yard run fumbled at the Southern Miss 4. cfbfastR had typed them as fumbles or penalties without a run or pass flag. Cleaning rule 4 recovers them.

No discrepancy is large enough to change a situation split.

## 8. The 2026 league environment

This section is league-wide and excludes the four matchup games. It shows whether early 2026 looks like early-season football in prior years, and it is the descriptive side of spec section 7.3. Weeks 1–2 are compared with Weeks 1–2 because early-season schedules are full of mismatches.

In [12]:
matchup_games = set(plays[(plays.season == config.CURRENT_SEASON) & ((plays.pos_team == config.OFFENSE_TEAM) | (plays.def_pos_team == config.DEFENSE_TEAM))].game_id)
early = model[model.week.isin(config.CURRENT_WEEKS) & model.game_season_type.eq("regular") & ~model.game_id.isin(matchup_games)].copy()  # bowls are also numbered week 1
early["fbs_vs_fbs"] = early.home_fbs & early.away_fbs
env = early.groupby(["fbs_vs_fbs", "season"]).agg(
    games=("game_id", "nunique"), plays=("play_key", "size"),
    ppa_per_play=("ppa", "mean"), explosive_rate=("explosive", "mean"), pass_rate=("is_pass", "mean"),
    garbage_time_share=("is_garbage_time", "mean"), spread_missing=("spread_missing", "mean"),
).round(3)
env.index = env.index.set_levels(["FBS vs. FCS/other", "FBS vs. FBS"], level=0)
display(env)

games  plays  ppa_per_play  explosive_rate  pass_rate  garbage_time_share  spread_missing
fbs_vs_fbs        season                                                                                           
FBS vs. FCS/other 2021       76   9350         0.214           0.067      0.486               0.227           0.000
                  2022       76   9537         0.200           0.067      0.482               0.237           0.024
                  2023       80   9934         0.204           0.068      0.485               0.218           0.000
                  2024       90  11238         0.192           0.068      0.481               0.231           0.175
                  2025       81  10208         0.178           0.065      0.478               0.233           0.000
                  2026       84  10370         0.190           0.067      0.467               0.291           0.000
FBS vs. FBS       2021       95  12494         0.186           0.068      0.496               0.107           0.000
                  2022      100  13221         0.176           0.068      0.510               0.116           0.000
                  2023       99  12647         0.174           0.068      0.518               0.104           0.000
                  2024       88  11180         0.168           0.067      0.499               0.115           0.000
                  2025       98  12675         0.184           0.068      0.502               0.106           0.000
                  2026       97  12270         0.183           0.064      0.496               0.119           0.000

**Reading the environment table.** In FBS-vs-FBS games, early 2026 sits inside the recent range for PPA per play, but it has fewer explosive plays and slightly more garbage time than regular-season Weeks 1–2 of 2021–2025. (Bowl and playoff games are also numbered week 1 in the play-by-play and are excluded here; spec change log #15.) These are league-wide shifts, not anything about Auburn or Florida. They're exactly why the baseline is checked against other 2026 plays before it scores the two teams, and corrected if the spec section 7.3 rule triggers. That check uses model residuals on Thursday; this table only describes raw rates.

## 9. How new are these teams?

Context only. None of these numbers enter a model.

In [13]:
coach_rows = []
for team in (config.OFFENSE_TEAM, config.DEFENSE_TEAM):
    for c in json.loads(cfbd_dest("coaches", {"team": team, "minYear": min(config.ALL_SEASONS), "maxYear": config.CURRENT_SEASON}).read_bytes()):
        for s in c["seasons"]:
            if s["school"] == team and s["year"] >= config.TEST_SEASON:
                coach_rows.append({"team": team, "season": s["year"], "coach": f"{c['firstName']} {c['lastName']}", "hired": c["hireDate"][:10], "record": f"{s['wins']}-{s['losses']}"})
display(pd.DataFrame(coach_rows).sort_values(["team", "season", "hired"]))

fbs = {t["school"] for t in json.loads(cfbd_dest("teams_fbs", {"year": config.CURRENT_SEASON}).read_bytes())}
ret = pd.DataFrame(json.loads(cfbd_dest("returning_production", {"year": config.CURRENT_SEASON}).read_bytes()))
ret = ret[ret.team.isin(fbs)].copy()
cols = {"percentPPA": "returning offensive PPA", "percentPassingPPA": "returning passing PPA", "usage": "returning usage"}
out = []
for team in (config.OFFENSE_TEAM, config.DEFENSE_TEAM):
    row = {"team": team}
    for c, label in cols.items():
        v = ret.set_index("team").loc[team, c]
        row[label] = f"{v:.1%} (rank {int((ret[c] > v).sum()) + 1} of {len(ret)})"
    out.append(row)
display(pd.DataFrame(out).set_index("team"))

portal = pd.DataFrame(json.loads(cfbd_dest("transfer_portal", {"year": config.CURRENT_SEASON}).read_bytes()))
portal = portal[portal.destination.isin(fbs)]
offense_pos, defense_pos = {"QB", "RB", "WR", "TE", "OT", "IOL", "ATH"}, {"DL", "EDGE", "LB", "CB", "S"}
portal["side"] = np.where(portal.position.isin(offense_pos), "offense", np.where(portal.position.isin(defense_pos), "defense", "specialists"))
by_side = portal.groupby(["destination", "side"]).size().unstack(fill_value=0)
summary = []
for team, side in ((config.OFFENSE_TEAM, "offense"), (config.DEFENSE_TEAM, "defense")):
    n = by_side.loc[team, side]
    summary.append({"team": team, "unit in this matchup": side, "incoming transfers": n,
                    "FBS rank": f"{int((by_side[side] > n).sum()) + 1} of {len(by_side)}", "FBS median": int(by_side[side].median())})
display(pd.DataFrame(summary).set_index("team"))

,team,season,coach,hired,record
1,Auburn,2025,Hugh Freeze,2022-11-29,4-5
0,Auburn,2025,D.J. Durkin,2025-11-02,1-2
2,Auburn,2026,Alex Golesh,2025-11-30,0-0
4,Florida,2025,Billy Napier,2021-11-28,3-4
3,Florida,2025,Billy Gonzales,2025-10-19,1-4
5,Florida,2026,Jon Sumrall,2025-11-30,0-0


,returning offensive PPA,returning passing PPA,returning usage
team,,,
Auburn,14.4% (rank 110 of 136),0.0% (rank 109 of 136),19.6% (rank 107 of 136)
Florida,69.1% (rank 23 of 136),94.3% (rank 32 of 136),79.0% (rank 12 of 136)


,unit in this matchup,incoming transfers,FBS rank,FBS median
team,,,,
Auburn,offense,27,2 of 135,10
Florida,defense,8,74 of 135,9


Auburn's offense is close to a brand-new unit: a first-year head coach, 14% of last season's offensive production returning, no returning passing production, and the second-largest offensive transfer class in FBS. Florida's defense has a first-year head coach and staff, but its defensive transfer intake was about average. CFBD publishes no defensive returning-production measure, so the roster turnover on Florida's defense can't be quantified further from this data. Either way, the rule holds: only 2026 plays describe these teams.

## 10. Team sample sizes (counts only)

These counts decide the cell structure before any residual exists (spec section 6). The primary team sample excludes garbage time.

In [14]:
current_model = model[model.season == config.CURRENT_SEASON]
auburn = current_model[current_model.pos_team == config.OFFENSE_TEAM]
florida = current_model[current_model.def_pos_team == config.DEFENSE_TEAM]

def game_counts(df, side_col):
    g = df.groupby(["week", side_col]).agg(model_sample=("play_key", "size"), garbage_time=("is_garbage_time", "sum"))
    g["team_profile"] = g.model_sample - g.garbage_time
    g["garbage share"] = (g.garbage_time / g.model_sample).map("{:.0%}".format)
    return g

print("Auburn offense by opponent"); display(game_counts(auburn, "def_pos_team"))
print("Florida defense by opponent"); display(game_counts(florida, "pos_team"))

Auburn offense by opponent


,,model_sample,garbage_time,team_profile,garbage share
week,def_pos_team,,,,
1,Baylor,75,0,75,0%
2,Southern Miss,82,23,59,28%


Florida defense by opponent


,,model_sample,garbage_time,team_profile,garbage share
week,pos_team,,,,
1,Florida Atlantic,89,28,61,31%
2,Campbell,61,27,34,44%


In [15]:
profile_a, profile_f = auburn[auburn.in_team_profile], florida[florida.in_team_profile]

def tiers(n):
    return np.select([n < config.MIN_PLAYS_DISPLAY, n < config.MIN_PLAYS_ELIGIBLE], ["hidden", "limited"], "eligible")

def cell_counts(keys):
    a = profile_a.groupby(keys).size().rename("Auburn offense")
    f = profile_f.groupby(keys).size().rename("Florida defense")
    t = pd.concat([a, f], axis=1).fillna(0).astype(int)
    t["Auburn tier"], t["Florida tier"] = tiers(t["Auburn offense"]), tiers(t["Florida defense"])
    t["both eligible"] = (t["Auburn tier"] == "eligible") & (t["Florida tier"] == "eligible")
    return t

fine = cell_counts(["play_family", "down_bin", "distance_bin"])
display(fine)

Auburn offense  Florida defense Auburn tier Florida tier  both eligible
play_family down_bin distance_bin                                                                         
pass        1st      long 8+                   25               21    eligible     eligible           True
            2nd      long 8+                   14                9     limited      limited          False
                     medium 4-7                 4                8      hidden      limited          False
                     short 1-3                  3                3      hidden       hidden          False
            3rd/4th  long 8+                    7                2      hidden       hidden          False
                     medium 4-7                 9               13     limited      limited          False
                     short 1-3                  3                5      hidden       hidden          False
run         1st      long 8+                   34               13    eligible      limited          False
                     medium 4-7                 1                0      hidden       hidden          False
            2nd      long 8+                    8                8     limited      limited          False
                     medium 4-7                 9                1     limited       hidden          False
                     short 1-3                  1                2      hidden       hidden          False
            3rd/4th  long 8+                    4                1      hidden       hidden          False
                     medium 4-7                 3                5      hidden       hidden          False
                     short 1-3                  9                3     limited       hidden          False
            1st      short 1-3                  0                1      hidden       hidden          False

In [16]:
profile_a = profile_a.assign(down_group=np.where(profile_a.is_passing_down, "passing downs", "standard downs"))
profile_f = profile_f.assign(down_group=np.where(profile_f.is_passing_down, "passing downs", "standard downs"))
rollup = cell_counts(["play_family", "down_group"])
display(rollup)

field = cell_counts(["play_family", "field_bin"])
display(field)

Auburn offense  Florida defense Auburn tier Florida tier  both eligible
play_family down_group                                                                             
pass        passing downs               25               21    eligible     eligible           True
            standard downs              40               40    eligible     eligible           True
run         passing downs               15               13    eligible      limited          False
            standard downs              54               21    eligible     eligible           True

Auburn offense  Florida defense Auburn tier Florida tier  both eligible
play_family field_bin                                                                                  
pass        midfield to opp 21              22               15    eligible     eligible           True
            own territory                   36               40    eligible     eligible           True
            red zone                         7                6      hidden       hidden          False
run         midfield to opp 21              24                3    eligible       hidden          False
            own territory                   36               24    eligible     eligible           True
            red zone                         9                7     limited       hidden          False

In [17]:
n_cells = len(fine)
both_eligible = int(fine["both eligible"].sum())
hidden_any = int(((fine["Auburn tier"] == "hidden") | (fine["Florida tier"] == "hidden")).sum())
decision = "down x distance x play family" if both_eligible > n_cells / 2 else "roll up to standard/passing downs x play family"
print(f"Fine cells: {n_cells}; eligible for both teams: {both_eligible}; hidden for at least one team: {hidden_any}")
print(f"Rollup cells eligible for both teams: {int(rollup['both eligible'].sum())} of {len(rollup)}")
print(f"Pre-registered cell structure for the main map: {decision}")
field_ok = bool(field["both eligible"].all())
print(f"Field position added to the main map: {'yes' if field_ok else 'no (some run or pass x field position cells are below the display threshold)'}")
gates["Cell structure chosen from counts before any residual"] = True

Fine cells: 16; eligible for both teams: 1; hidden for at least one team: 11
Rollup cells eligible for both teams: 3 of 4
Pre-registered cell structure for the main map: roll up to standard/passing downs x play family
Field position added to the main map: no (some run or pass x field position cells are below the display threshold)


Per spec section 6, the fine down × distance × play family grid is used for the main map only if most of its cells are eligible for both teams. Otherwise the main map rolls up to standard and passing downs × run and pass, and fine cells that meet the display threshold appear only in supporting heatmaps with their sample-size labels. The printed decision above is binding for Thursday.

## 11. Data-quality gates

In [18]:
gate_table = pd.DataFrame({"gate": list(gates), "passed": list(gates.values())})
display(gate_table)
print("ALL GATES PASSED" if gate_table.passed.all() else "SOME GATES FAILED")

,gate,passed
0,All six seasons load with every required column,True
1,play_key unique after cleaning,True
2,PPA identical for identical states across 2021-2026,True
3,No required field >10% missing for Auburn offense or Florida defense,True
4,All four final scores match official records,True
5,"Plays, pass attempts, and interceptions within 1 of official for every offense",True
6,Frozen PPA equals CFBD on every non-turnover play,True
7,Cell structure chosen from counts before any residual,True


ALL GATES PASSED
